# 02 – Data Preprocessing and Feature Engineering

## 1. Introduction

Data preprocessing is an important step in building the Smart Traffic Risk Prediction System. It helps us understand the dataset's structure, identify missing values and duplicate records, and prepare the data for machine learning.

In this notebook, we will:

* Load the accident dataset.
* Inspect the dataset dimensions and column data types.
* Identify missing values.
* Check for duplicate records.
* Examine accident severity labels.
* Prepare the dataset for feature engineering and model training.

In [1]:
import pandas as pd
import numpy as np

# Load the original dataset
file_path = "C:/Minor Projects/dataa/raw/Accident_data.csv"
df = pd.read_csv(file_path)

# Create a working copy for preprocessing
df_clean = df.copy()

# Check dataset size
print("Dataset shape:", df_clean.shape)

# Check missing values
print("\nMissing values:")
print(df_clean.isnull().sum())

# Check duplicate rows
print("\nDuplicate rows:", df_clean.duplicated().sum())

# Check data types
print("\nData types:")
print(df_clean.dtypes)

Dataset shape: (8116, 13)

Missing values:
Date                                           0
Day_of_Week                                    0
Time_of_Accident                               0
Accident_Location_A                            0
Accident_Location_A_Chainage_km                0
Accident_Location_A_Chainage_km_RoadSide       0
Accident_Severity_C                            0
Causes_D                                       0
Road_Feature_E                                 0
Road_Condition_F                               0
Weather_Conditions_H                           0
Vehicle_Type_Involved_J_V1                     0
Vehicle_Type_Involved_J_V2                  6154
dtype: int64

Duplicate rows: 4063

Data types:
Date                                            str
Day_of_Week                                   int64
Time_of_Accident                                str
Accident_Location_A                           int64
Accident_Location_A_Chainage_km             float64
Accident_Loc

## 2. Accident Severity Analysis

Accident severity is an important variable for the traffic risk prediction system. Before preparing the target variable for machine learning, we need to inspect the severity values present in the dataset.

The original severity codes must be interpreted using the dataset documentation before assigning risk categories.


In [2]:

# Inspect accident severity values
print("Accident severity distribution:")
print(
    df_clean["Accident_Severity_C"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nSeverity data type:")
print(df_clean["Accident_Severity_C"].dtype)

Accident severity distribution:
Accident_Severity_C
1     708
2    2904
3    3088
4    1416
Name: count, dtype: int64

Severity data type:
int64


## 3. Missing Value and Duplicate Record Assessment

Missing values and duplicate records can affect data quality. We will examine their frequency before deciding how to handle them. Missing vehicle information will not automatically be treated as an error because it may indicate that information about a second vehicle was not recorded.

In [3]:
# Summarize missing values and unique values
missing_summary = pd.DataFrame({
    "Missing_Count": df_clean.isnull().sum(),
    "Missing_Percentage": (df_clean.isnull().mean() * 100).round(2),
    "Unique_Values": df_clean.nunique()
})

display(
    missing_summary.sort_values(
        by="Missing_Percentage",
        ascending=False
    )
)

# Count duplicate records
print("Duplicate rows:", df_clean.duplicated().sum())

,Missing_Count,Missing_Percentage,Unique_Values
Vehicle_Type_Involved_J_V2,6154,75.83,15
Day_of_Week,0,0.00,7
Time_of_Accident,0,0.00,797
Accident_Location_A,0,0.00,2
Date,0,0.00,1664
Accident_Location_A_Chainage_km,0,0.00,2569
Accident_Location_A_Chainage_km_RoadSide,0,0.00,2
Causes_D,0,0.00,7
Accident_Severity_C,0,0.00,4
Road_Feature_E,0,0.00,4


Duplicate rows: 4063


## 4. Detailed Inspection of Important Features

In this section, we inspect the values in important accident-related columns, including severity, weather conditions, road conditions, road features, causes, and vehicle types. This helps us identify inconsistent values and understand how the features should be prepared for machine learning.

In [4]:
columns_to_check = [
    "Accident_Severity_C",
    "Weather_Conditions_H",
    "Road_Condition_F",
    "Road_Feature_E",
    "Causes_D",
    "Vehicle_Type_Involved_J_V1",
    "Vehicle_Type_Involved_J_V2"
]

for column in columns_to_check:
    print(f"\n--- {column} ---")
    print(df_clean[column].value_counts(dropna=False).head(20))


--- Accident_Severity_C ---
Accident_Severity_C
3    3088
2    2904
4    1416
1     708
Name: count, dtype: int64

--- Weather_Conditions_H ---
Weather_Conditions_H
1     5334
2     1198
3      458
10     440
9      190
4      182
8       96
5       88
11      66
12      48
7       14
6        2
Name: count, dtype: int64

--- Road_Condition_F ---
Road_Condition_F
1    6800
2     570
4     490
3     144
5      36
8      34
6      30
7      12
Name: count, dtype: int64

--- Road_Feature_E ---
Road_Feature_E
4    6860
2    1190
1      58
3       8
Name: count, dtype: int64

--- Causes_D ---
Causes_D
2    3810
3    2968
4     682
1     474
5     176
8       4
6       2
Name: count, dtype: int64

--- Vehicle_Type_Involved_J_V1 ---
Vehicle_Type_Involved_J_V1
6     2190
5     2142
1     1858
11     722
9      304
3      214
14     164
8      146
7      108
15      94
10      92
2       44
4       20
12      18
Name: count, dtype: int64

--- Vehicle_Type_Involved_J_V2 ---
Vehicle_Type_Involve

## 5. Date and Time Data Inspection

The accident date and time may help identify temporal patterns in accident occurrence. We will inspect their formats before extracting features such as month, day of the week, and hour of the day.

In [5]:
# Inspect date and time formats before converting them
print("Sample dates:")
print(df_clean["Date"].head(10))

print("\nSample accident times:")
print(df_clean["Time_of_Accident"].head(10))

print("\nDate column data type:", df_clean["Date"].dtype)
print("Time column data type:", df_clean["Time_of_Accident"].dtype)

Sample dates:
0      1/9/2013
1      4/9/2013
2      9/9/2013
3     11/9/2013
4    17/09/2013
5    19/09/2013
6    19/09/2013
7    20/09/2013
8    26/09/2013
9    27/09/2013
Name: Date, dtype: str

Sample accident times:
0    13:00
1    15:25
2     0:40
3     7:30
4    13:00
5    10:15
6    23:40
7     8:15
8    22:15
9    18:25
Name: Time_of_Accident, dtype: str

Date column data type: str
Time column data type: str


## 6. Date and Time Conversion

The date and time columns are currently stored as strings. In this step, we convert them into datetime formats so that meaningful temporal features can be extracted for accident risk prediction.

The date is parsed using day-first interpretation, consistent with the date examples in the dataset. Invalid or unrecognized values are identified for further inspection. The original columns are preserved.

In [6]:
# Convert date and time into appropriate formats
df_clean["Date_parsed"] = pd.to_datetime(
    df_clean["Date"],
    dayfirst=True,
    errors="coerce"
)

df_clean["Time_parsed"] = pd.to_datetime(
    df_clean["Time_of_Accident"],
    format="%H:%M",
    errors="coerce"
)

# Check whether any values failed to convert
print("Invalid dates:", df_clean["Date_parsed"].isna().sum())
print("Invalid times:", df_clean["Time_parsed"].isna().sum())

# Display the original and converted values
display(
    df_clean[
        ["Date", "Date_parsed", "Time_of_Accident", "Time_parsed"]
    ].head(10)
)

Invalid dates: 0
Invalid times: 0


,Date,Date_parsed,Time_of_Accident,Time_parsed
0,1/9/2013,2013-09-01,13:00,1900-01-01 13:00:00
1,4/9/2013,2013-09-04,15:25,1900-01-01 15:25:00
2,9/9/2013,2013-09-09,0:40,1900-01-01 00:40:00
3,11/9/2013,2013-09-11,7:30,1900-01-01 07:30:00
4,17/09/2013,2013-09-17,13:00,1900-01-01 13:00:00
5,19/09/2013,2013-09-19,10:15,1900-01-01 10:15:00
6,19/09/2013,2013-09-19,23:40,1900-01-01 23:40:00
7,20/09/2013,2013-09-20,8:15,1900-01-01 08:15:00
8,26/09/2013,2013-09-26,22:15,1900-01-01 22:15:00
9,27/09/2013,2013-09-27,18:25,1900-01-01 18:25:00


## 7. Temporal Feature Extraction

After converting the date and time fields, we extract useful temporal features, including the accident year, month, day of the week, and hour. These features can help a machine learning model learn patterns associated with different times and days.

The extracted features will be checked before proceeding to further preprocessing.

In [7]:
# Extract temporal features
df_clean["Accident_Year"] = df_clean["Date_parsed"].dt.year
df_clean["Accident_Month"] = df_clean["Date_parsed"].dt.month
df_clean["Accident_DayOfWeek"] = df_clean["Date_parsed"].dt.dayofweek
df_clean["Accident_Hour"] = df_clean["Time_parsed"].dt.hour

# Display the newly created features
display(
    df_clean[
        [
            "Date_parsed",
            "Accident_Year",
            "Accident_Month",
            "Accident_DayOfWeek",
            "Accident_Hour"
        ]
    ].head(10)
)

# Check missing values in the extracted features
print("\nMissing values in temporal features:")
print(
    df_clean[
        [
            "Accident_Year",
            "Accident_Month",
            "Accident_DayOfWeek",
            "Accident_Hour"
        ]
    ].isna().sum()
)

,Date_parsed,Accident_Year,Accident_Month,Accident_DayOfWeek,Accident_Hour
0,2013-09-01,2013,9,6,13
1,2013-09-04,2013,9,2,15
2,2013-09-09,2013,9,0,0
3,2013-09-11,2013,9,2,7
4,2013-09-17,2013,9,1,13
5,2013-09-19,2013,9,3,10
6,2013-09-19,2013,9,3,23
7,2013-09-20,2013,9,4,8
8,2013-09-26,2013,9,3,22
9,2013-09-27,2013,9,4,18



Missing values in temporal features:
Accident_Year         0
Accident_Month        0
Accident_DayOfWeek    0
Accident_Hour         0
dtype: int64


## 8. Feature Engineering

Feature engineering transforms the original accident data into useful inputs for machine learning. In this step, we prepare temporal features and retain relevant accident characteristics, including weather conditions, road conditions, accident causes, and vehicle types.

The original dataset columns and accident severity labels are preserved. The resulting dataset will be checked before being saved for model development.

In [8]:
# Create additional temporal features
df_clean["Is_Weekend"] = (
    df_clean["Accident_DayOfWeek"] >= 5
).astype("Int64")

# Inspect the engineered features
feature_columns = [
    "Date_parsed",
    "Time_parsed",
    "Accident_Year",
    "Accident_Month",
    "Accident_DayOfWeek",
    "Accident_Hour",
    "Is_Weekend",
    "Accident_Severity_C",
    "Weather_Conditions_H",
    "Road_Condition_F"
]

display(df_clean[feature_columns].head(10))

print("Dataset shape after feature engineering:", df_clean.shape)

,Date_parsed,Time_parsed,Accident_Year,Accident_Month,Accident_DayOfWeek,Accident_Hour,Is_Weekend,Accident_Severity_C,Weather_Conditions_H,Road_Condition_F
0,2013-09-01,1900-01-01 13:00:00,2013,9,6,13,1,3,8,3
1,2013-09-04,1900-01-01 15:25:00,2013,9,2,15,0,4,12,6
2,2013-09-09,1900-01-01 00:40:00,2013,9,0,0,0,4,12,6
3,2013-09-11,1900-01-01 07:30:00,2013,9,2,7,0,3,8,3
4,2013-09-17,1900-01-01 13:00:00,2013,9,1,13,0,4,7,4
5,2013-09-19,1900-01-01 10:15:00,2013,9,3,10,0,3,8,4
6,2013-09-19,1900-01-01 23:40:00,2013,9,3,23,0,2,12,6
7,2013-09-20,1900-01-01 08:15:00,2013,9,4,8,0,2,12,4
8,2013-09-26,1900-01-01 22:15:00,2013,9,3,22,0,3,12,5
9,2013-09-27,1900-01-01 18:25:00,2013,9,4,18,0,3,8,3


Dataset shape after feature engineering: (8116, 20)


## 9. Save the Prepared Dataset

After inspecting the converted date and time fields and the engineered features, save a working copy of the dataset in the processed-data directory. This preserves the original raw CSV and allows the model-training notebook to load the prepared data later.

In [9]:
from pathlib import Path

# Create the processed-data directory if needed
processed_dir = Path("C:/Minor Projects/dataa/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# Save the prepared dataset
output_path = processed_dir / "accident_data_prepared.csv"
df_clean.to_csv(output_path, index=False)

print("Prepared dataset saved to:", output_path)
print("Saved dataset shape:", df_clean.shape)

Prepared dataset saved to: C:\Minor Projects\dataa\processed\accident_data_prepared.csv
Saved dataset shape: (8116, 20)
